In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import SimpleITK
import itk
import json
import SimpleITK as sitk
import time

In [2]:
info_file = '../../data/dual_energy_submission_label/label_submission_new.csv'
df_info = pd.read_csv(info_file)
key_name = 'DualEnergy BoneRemovalHead Suppressed'
df_info.head()
BoneRemovalHead_dict = {}
BoneRemovalHead_vr_dict = {}
for index, row in df_info.iterrows():
    if key_name in row['info']:
        BoneRemovalHead_dict[row['seriesInstanceUID']] = row['studyInstanceUID']

In [3]:
vr_dict = {}
mip_dict = {}
removebone_dict = {}
highvoltage_dict = {}
lowvoltage_dict = {}
mix_dict = {}
study_removebone_dict = {}

vr_key_name = 'VRT Collection'
mip_key_name = 'MIP Collection'
removebone_key_name = 'DualEnergy BoneRemovalHead Suppressed'
highvoltage_key_name = 'A_140kV'
lowvoltage_key_name = 'B_80kV'
mix_key_name = 'M_0.3'


for index, row in df_info.iterrows():
    if vr_key_name in row['info']:
        vr_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
    elif mip_key_name in row['info']:
        mip_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
    elif removebone_key_name in row['info']:
        removebone_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
        study_removebone_dict[row['seriesInstanceUID']] = row['studyInstanceUID']
    elif highvoltage_key_name in row['info']:
        highvoltage_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
    elif lowvoltage_key_name in row['info']:
        lowvoltage_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
    elif mix_key_name in row['info']:
        mix_dict[row['studyInstanceUID']] = row['seriesInstanceUID']
        
# print(len(vr_dict))
# print(len(mip_dict))
# print(len(removebone_dict))
# print(len(highvoltage_dict))
# print(len(lowvoltage_dict))
# print(len(mix_dict))

outdir = 'config'
os.makedirs(outdir, exist_ok=True)

with open(os.path.join(outdir, 'vr_dict.json'), 'w') as f:
    f.write(json.dumps(vr_dict))
    
with open(os.path.join(outdir, 'mip_dict.json'), 'w') as f:
    f.write(json.dumps(mip_dict))
    
with open(os.path.join(outdir, 'removebone_dict.json'), 'w') as f:
    f.write(json.dumps(removebone_dict))
    
with open(os.path.join(outdir, 'highvoltage_dict.json'), 'w') as f:
    f.write(json.dumps(highvoltage_dict))
    
with open(os.path.join(outdir, 'lowvoltage_dict.json'), 'w') as f:
    f.write(json.dumps(lowvoltage_dict))
    
with open(os.path.join(outdir, 'mix_dict.json'), 'w') as f:
    f.write(json.dumps(mix_dict))
    
with open(os.path.join(outdir, 'study_removebone_dict.json'), 'w') as f:
    f.write(json.dumps(study_removebone_dict))
    
root_dir = '/data/zhangwd/data/examples/brain/bystudy'

In [4]:
root_dir = '/data/zhangwd/data/examples/brain/bystudy'
# return study, vr, mip, hv, lv, mix, removebone
def get_sub_series(in_series_uid_removed_bone, data_dir):
    study_uid = study_removebone_dict[in_series_uid_removed_bone]
    vr_series_uid = os.path.join(data_dir, study_uid, vr_dict[study_uid])
    mip_series_uid = os.path.join(data_dir, study_uid, mip_dict[study_uid])
    hv_series_uid = os.path.join(data_dir, study_uid, highvoltage_dict[study_uid])
    lv_series_uid = os.path.join(data_dir, study_uid, lowvoltage_dict[study_uid])
    mix_series_uid = os.path.join(data_dir, study_uid, mix_dict[study_uid])
    removebone_series_uid = os.path.join(data_dir, study_uid, removebone_dict[study_uid])
    return study_uid, vr_series_uid, mip_series_uid, hv_series_uid, lv_series_uid, mix_series_uid, removebone_series_uid

In [5]:
series_list_file = '/data/zhangwd/data/examples/brain/bone_removed/removed_dicom.txt'
root_dir = '/data/zhangwd/data/examples/brain/bystudy'
series_uids = []
with open(series_list_file) as f:
    for line in f.readlines():
        line = line.strip()
        if line is None or len(line) == 0:
            continue
        series_uids.append(line)
        
removebone_series_uid = os.path.basename(series_uids[1])
print(removebone_series_uid)
study_uid, vr_series_uid,mip_series_uid, hv_series_uid, lv_series_uid, mix_series_uid, removebone_series_uid =  get_sub_series(removebone_series_uid, root_dir)

1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240


In [6]:
print(hv_series_uid)
print(removebone_series_uid)

/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.1.4.60320.30000017032023565187200001259
/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240


In [7]:
def get_delta_img(series_uid1, series_uid2, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    reader = sitk.ImageSeriesReader()
    filenamesDICOM = reader.GetGDCMSeriesFileNames(series_uid1)
    print(len(filenamesDICOM))
    reader.SetFileNames(filenamesDICOM)
    imgOriginal1 = reader.Execute()
    
    # remove bone volume data
    reader = sitk.ImageSeriesReader()
    filenamesDICOM = reader.GetGDCMSeriesFileNames(series_uid2)
    print(len(filenamesDICOM))
    reader.SetFileNames(filenamesDICOM)
    imgOriginal2 = reader.Execute()
    
    
    substract_filter = sitk.SubtractImageFilter()
    substract_img = substract_filter.Execute(imgOriginal1, imgOriginal2)
    
    threshold_filter = sitk.BinaryThresholdImageFilter()
    threshold_filter.SetInsideValue(1)
    threshold_filter.SetOutsideValue(0)
    threshold_filter.SetLowerThreshold(20)
    threshold_filter.SetUpperThreshold(4095)
    threshold_img = threshold_filter.Execute(substract_img)
    
    erode_filter = sitk.BinaryErodeImageFilter()
    erode_filter.SetKernelRadius(2)
    erode_img = erode_filter.Execute(threshold_img)
    
    dilation_filter = sitk.BinaryDilateImageFilter()
    dilation_filter.SetKernelRadius(4)
    dilation_img = dilation_filter.Execute(erode_img)
    
    
    
#     # for vessel
#     wl = 287
#     ww = 332
#     lower = int(wl-ww/2)
#     upper = int(wl+ww/2)
#     threshold_filter = sitk.BinaryThresholdImageFilter()
#     threshold_filter.SetInsideValue(1)
#     threshold_filter.SetOutsideValue(0)
#     threshold_filter.SetLowerThreshold(lower)
#     threshold_filter.SetUpperThreshold(2000)
#     vessel_img = threshold_filter.Execute(imgOriginal2)
    
#     erode_filter = sitk.BinaryErodeImageFilter()
#     erode_filter.SetKernelRadius(1)
#     vessel_erode_img = erode_filter.Execute(vessel_img)
    
#     dilation_filter = sitk.BinaryDilateImageFilter()
#     dilation_filter.SetKernelRadius(1)
#     vessel_dilation_img = dilation_filter.Execute(vessel_erode_img)
    
#     substract_filter = sitk.AddImageFilter()
#     vessel_substract_img = substract_filter.Execute(vessel_dilation_img, dilation_img)
    
#     threshold_filter = sitk.BinaryThresholdImageFilter()
#     threshold_filter.SetInsideValue(1)
#     threshold_filter.SetOutsideValue(0)
#     threshold_filter.SetLowerThreshold(1)
#     threshold_filter.SetUpperThreshold(1)
#     vessel_better_img = threshold_filter.Execute(vessel_substract_img)
    
    sitk.WriteImage(imgOriginal1 ,os.path.join(out_dir, 'volume.nii'))
    sitk.WriteImage(threshold_img, os.path.join(out_dir, 'threshold_img.nii.gz'))
    sitk.WriteImage(erode_img, os.path.join(out_dir, 'erode_img.nii.gz'))
    sitk.WriteImage(dilation_img, os.path.join(out_dir, 'dilation_img.nii'))
    sitk.WriteImage(substract_img, os.path.join(out_dir, 'substract_img.nii.gz'))
#     sitk.WriteImage(vessel_img, os.path.join(out_dir, 'vessel_img.nii.gz'))
#     sitk.WriteImage(vessel_erode_img, os.path.join(out_dir, 'vessel_erode_img.nii.gz'))
#     sitk.WriteImage(vessel_dilation_img, os.path.join(out_dir, 'vessel_dilation_img.nii.gz'))
#     sitk.WriteImage(vessel_substract_img, os.path.join(out_dir, 'vessel_substract_img.nii.gz'))
#     sitk.WriteImage(vessel_better_img, os.path.join(out_dir, 'vessel_better_img.nii.gz'))
    
    return substract_img


def check_validation(series_uid1, series_uid2):
    reader = sitk.ImageSeriesReader()
    filenamesDICOM1 = reader.GetGDCMSeriesFileNames(series_uid1)
    filenamesDICOM2 = reader.GetGDCMSeriesFileNames(series_uid2)
    if len(filenamesDICOM1) == len(filenamesDICOM2):
        return True
    return False

In [8]:
# hv_delta_img = get_delta_img(hv_series_uid, removebone_series_uid, './delta/{}'.format(os.path.basename(removebone_series_uid)))
# lv_delta_img = get_delta_img(lv_series_uid, removebone_series_uid)
# mix_delta_img = get_delta_img(mix_series_uid, removebone_series_uid)

In [9]:
# os.makedirs('./delta', exist_ok=True)
# sitk.WriteImage(delta_img, './delta/hv_delta_.nii.gz')
# sitk.WriteImage(delta_img, './delta/lv_delta_.nii.gz')
# sitk.WriteImage(delta_img, './delta/mix_delta_.nii.gz')

In [10]:
# ?sitk.SubtractImageFilter

In [11]:
record_list = []
for series_uid in series_uids:
    try:
        removebone_series_uid = os.path.basename(series_uid)
        study_uid, vr_series_uid,mip_series_uid, hv_series_uid, lv_series_uid, mix_series_uid, removebone_series_uid =  get_sub_series(removebone_series_uid, root_dir)
        if not check_validation(hv_series_uid, removebone_series_uid):
            continue
        print('====> processing {} and {}'.format(hv_series_uid, removebone_series_uid))
        beg = time.time()
        hv_delta_img = get_delta_img(hv_series_uid, removebone_series_uid, './delta/{}'.format(os.path.basename(removebone_series_uid)))
        record_list.append('{}\t{}\{}'.format(hv_series_uid, removebone_series_uid, './delta/{}'.format(os.path.basename(removebone_series_uid))))
        end = time.time()
        print('time elapsed:{:.3f}'.format(end-beg))
    except:
        continue

====> processing /data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.1.4.60320.30000016072823593314100002689 and /data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799
339
339
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.1.4.60320.30000017032023565187200001259 and /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240
371
371
time elapsed:64.771
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.1.4.60320.30000018101623535580200010428 and /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128
389
389
time elapsed:67.446
====> process

342
342
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.2017050900232223/1.3.12.2.1107.5.1.4.60320.30000017050923454540200002623 and /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.2017050900232223/1.3.12.2.1107.5.99.2.9594.30000017051013402768700000128
337
337
time elapsed:70.027
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190425031811807/1.3.12.2.1107.5.1.4.60320.30000019042423372139300008933 and /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190425031811807/1.3.12.2.1107.5.99.2.9594.30000019042616245185900000128
453
453
time elapsed:89.906
====> processing /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190413003611840/1.3.12.2.1107.5.1.4.60320.30000019041223565887400000499 and /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190413003611840/1.3.12.2.1107.5.99.2.9594.30000019041407443398400000128
456
456
time elapsed:89.797


In [13]:
len(record_list)

22

In [16]:
ls ./config

highvoltage_dict.json  mip_dict.json  removebone_dict.json        vr_dict.json
lowvoltage_dict.json   mix_dict.json  study_removebone_dict.json


In [17]:
pwd

'/home/zhangwd/code/work/BrainSolution/ct_dual_energy_substraction/itk_algo'

In [71]:
col_a_list = []
col_b_list = []
col_c_list = []
col_d_list = []
study_uids = []
removebone_list = []
import os
print(root_dir)
for elem in record_list:
    ss = elem.split('\t')
    ss = ss[1].split('\\')
    removebone_list.append(ss[0])
    removebone_series_uid1 = os.path.basename(ss[0])
    try:
        study_uid, vr_series_uid,mip_series_uid, hv_series_uid, lv_series_uid, mix_series_uid, removebone_series_uid =  get_sub_series(removebone_series_uid1, root_dir)
    except:
        continue
    col_a_list.append(os.path.basename(removebone_series_uid))
    col_b_list.append(os.path.basename(hv_series_uid))
    col_c_list.append(os.path.basename(lv_series_uid))
    col_d_list.append(os.path.basename(mix_series_uid))
    study_uids.append(os.path.basename(study_uid))   

/data/zhangwd/data/examples/brain/bystudy


In [72]:
import pandas as pd
zippedList =  list(zip(study_uids, col_a_list, col_b_list, col_c_list, col_d_list, removebone_list))
df = pd.DataFrame(zippedList, columns=['study_uid', '剪影', 'high voltage', 'low voltage', 'mix', 'full path'])
df.to_csv('./config/out.csv')

In [73]:
pwd

'/home/zhangwd/code/work/BrainSolution/ct_dual_energy_substraction/itk_algo'

In [77]:
with open('./config/full_path.txt', 'w') as f:
    f.write('\n'.join(removebone_list))

In [75]:
removebone_list

['/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240',
 '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128',
 '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160408005129972/1.3.12.2.1107.5.99.2.9594.30000016040823144126500004946',
 '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190505015817095/1.3.12.2.1107.5.99.2.9594.30000019042916441620300004106',
 '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190401064814798/1.3.12.2.1107.5.99.2.9594.30000019033013292326500013126',
 '/data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160426001922888/1.3.12.2.1107.5.99.2.9594.30000016042715412376500000294',
 '/data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016101900120053800000005/1.3.12.2.1107.5.99.2.9594.3000001610181241538430